# ⚡ ALOHA 14-DOF Bimanual Dynamic Toss & Catch // Cloud High-Speed Training Pipeline

> **화면 렌더링 오버헤드를 100% 제거하고, 순수 물리 동역학 연산과 병렬 강화학습(PPO)에 집중하여 수십만 스텝을 단 몇 분 만에 초고속 훈련하는 클라우드 전용 파이프라인**입니다.

---

### 💡 클라우드 훈련과 로컬 시각화의 최적 역할 분담 (Architecture)
1. **Google Colab (초고속 두뇌 훈련 공장):**
   * 화면을 그리는 그래픽 오버헤드 없이, 8개 이상의 병렬 환경에서 순수 물리 수식만 초당 수만 스텝으로 연산하여 가장 똑똑한 **정책 가중치 파일(`aloha_toss_catch_ppo.zip`, 약 3MB)**을 빠르게 생산합니다.
2. **로컬 컴퓨터 (60fps 텔레메트리 콕핏 스튜디오):**
   * 코랩에서 생산된 `.zip` 가중치 파일 하나만 내려받아, 로컬 PC의 `evaluate.py`로 60fps 사이버 콕핏 HUD가 입혀진 최고 화질의 영상과 GIF를 추출합니다.

---  
## 🛠️ Step 1. 필수 라이브러리 설치 & 연산 가속 환경 준비

### 📖 왜 이 단계를 수행하나요?
* **MuJoCo 3.x:** 구글 딥마인드의 고정밀 접촉 역학 물리 엔진으로, 복잡한 다관절 로봇의 충돌과 탄도 궤적을 실시간보다 수십 배 빠르게 연산합니다.
* **Gymnasium:** 강화학습 표준 환경 인터페이스(`reset`, `step`)를 제공합니다.
* **Stable-Baselines3 & PyTorch:** 연속 제어 최신 강화학습 알고리즘인 **PPO (Proximal Policy Optimization)**의 멀티 프로세스 벡터화 훈련을 지원합니다.
* 화면 렌더링 패키지를 배제하고 순수 연산 라이브러리만 설치하므로 세팅이 가볍고 즉각 완료됩니다.

👉 **[실행 방법]** 아래 셀을 클릭하고 `Shift + Enter`를 누르세요.

In [ ]:
# 순수 연산 및 병렬 강화학습 필수 패키지 설치
!pip install -q mujoco gymnasium stable-baselines3 torch

import os
import mujoco
import gymnasium as gym
import torch
import numpy as np

print("===================================================")
print(f"✅ MuJoCo 엔진 버전   : {mujoco.__version__}")
print(f"✅ PyTorch 프레임워크  : {torch.__version__} (CUDA GPU 가속: {torch.cuda.is_available()})")
print(f"✅ Gymnasium 버전      : {gym.__version__}")
print("===================================================")

---  
## 🏗️ Step 2. 14자유도 듀얼 로봇 팔 물리 모델 (MJCF XML) 작성

### 📖 왜 이 단계를 수행하나요?
시뮬레이터 내부의 물리 법칙(중력, 마찰력, 관절 토크 한계값)을 MuJoCo 전용 XML로 정의하는 단계입니다.

* **좌측 투척 팔 (7-DOF):** 물체를 안정적으로 쥐고 공중으로 쏘아 올리는 고토크 구동기 및 서보 그리퍼 장착
* **우측 캐칭 팔 (7-DOF):** 날아오는 물체의 충격을 흡수하도록 높은 감쇠(`damping`)와 마찰 계수(`friction=2.0`)가 적용된 완충 패들 및 접촉 센서 장착
* **비행 캡슐 (Projectile):** 6자유도 자유 관절(`freejoint`)을 통해 포물선 비행 동역학 구현

👉 **[실행 방법]** 아래 셀을 실행하여 물리 모델 파일을 생성하고 MuJoCo 엔진에 탑재합니다.

In [ ]:
os.makedirs("assets", exist_ok=True)

xml_content = """<mujoco model="aloha_bimanual_toss_catch">
  <compiler angle="radian" meshdir="assets" autolimits="true"/>

  <option integrator="Euler" timestep="0.002" gravity="0 0 -9.81">
    <flag contact="enable" energy="enable"/>
  </option>

  <visual>
    <headlight ambient="0.15 0.18 0.22" diffuse="0.8 0.85 0.9" specular="0.3 0.3 0.3"/>
    <quality shadowsize="4096"/>
    <global azimuth="140" elevation="-25" offwidth="1920" offheight="1080"/>
  </visual>

  <asset>
    <!-- Dark Studio Aesthetics & Textures -->
    <texture type="skybox" builtin="gradient" rgb1="0.03 0.04 0.06" rgb2="0.08 0.1 0.14" width="512" height="512"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.05 0.06 0.08" rgb2="0.08 0.09 0.12" markrgb="0.15 0.2 0.28" width="500" height="500"/>
    <material name="groundmat" texture="groundplane" texuniform="true" texrepeat="15 15" reflectance="0.18" roughness="0.4"/>
    
    <material name="dark_carbon" rgba="0.12 0.13 0.15 1.0" specular="0.8" shininess="0.9" roughness="0.2"/>
    <material name="anodized_cyan" rgba="0.0 0.75 0.9 1.0" specular="0.9" shininess="0.9"/>
    <material name="anodized_magenta" rgba="0.95 0.15 0.45 1.0" specular="0.9" shininess="0.9"/>
    <material name="matte_silver" rgba="0.75 0.78 0.82 1.0" specular="0.6" roughness="0.3"/>
    <material name="neon_target" rgba="0.0 1.0 0.6 0.85" emission="0.6"/>
    <material name="neon_trajectory" rgba="0.2 0.8 1.0 0.6" emission="0.8"/>
    <material name="projectile_mat" rgba="1.0 0.4 0.1 1.0" specular="0.9" shininess="0.9" emission="0.2"/>
    <material name="cushion_pad" rgba="0.1 0.7 0.4 1.0" roughness="0.9" reflectance="0.05"/>
  </asset>

  <default>
    <joint limited="true" damping="1.5" armature="0.08" frictionloss="0.1"/>
    <geom contype="1" conaffinity="1" condim="4" friction="1.2 0.01 0.001" solref="0.005 1" solimp="0.95 0.99 0.001"/>
    <default class="thrower">
      <geom material="dark_carbon"/>
    </default>
    <default class="catcher">
      <geom material="dark_carbon"/>
    </default>
    <default class="cushion_geom">
      <geom material="cushion_pad" friction="2.0 0.05 0.005" solref="0.02 1.5" solimp="0.8 0.95 0.001"/>
    </default>
  </default>

  <worldbody>
    <light directional="true" pos="-0.5 1.5 3.0" dir="0.2 -0.5 -1" diffuse="0.9 0.9 0.95" specular="0.4 0.4 0.4" castshadow="true"/>
    <light directional="true" pos="1.5 -1.0 2.5" dir="-0.5 0.3 -1" diffuse="0.3 0.4 0.6" specular="0.2 0.2 0.3"/>
    
    <!-- Arena Ground Floor -->
    <geom name="floor" type="plane" size="3 3 0.1" material="groundmat"/>
    <geom name="table" type="box" pos="0 0 0.35" size="0.9 0.5 0.35" rgba="0.08 0.09 0.12 1.0" material="dark_carbon" solref="0.001 1" solimp="0.99 0.999 0.001"/>
    <geom name="table_trim" type="box" pos="0 0 0.705" size="0.91 0.51 0.005" material="anodized_cyan" solref="0.001 1" solimp="0.99 0.999 0.001"/>

    <!-- Arena Target Flush Visualization Sites (Flush with table surface, no floating cylinder clipping) -->
    <site name="toss_launch_zone" pos="-0.25 0 0.712" size="0.08 0.08 0.001" type="cylinder" material="anodized_magenta"/>
    <site name="toss_apex_zone" pos="0.0 0 1.25" size="0.04" type="sphere" material="neon_trajectory"/>
    <site name="catch_target_zone" pos="0.25 0 0.712" size="0.09 0.09 0.001" type="cylinder" material="neon_target"/>

    <!-- ============================================================== -->
    <!-- LEFT ARM: 7-DOF HIGH-SPEED THROWER (Base at X = -0.50m)        -->
    <!-- ============================================================== -->
    <body name="left_base" pos="-0.50 0 0.71">
      <geom name="left_pedestal" type="cylinder" size="0.07 0.05" pos="0 0 0.05" material="dark_carbon" solref="0.002 1" solimp="0.98 0.995 0.001"/>
      <geom name="left_accent" type="cylinder" size="0.072 0.005" pos="0 0 0.10" material="anodized_magenta" contype="0" conaffinity="0"/>
      
      <!-- J1: Waist / Yaw -->
      <body name="left_link1" pos="0 0 0.105">
        <joint name="left_waist" type="hinge" axis="0 0 1" range="-2.6 2.6"/>
        <geom type="cylinder" size="0.058 0.07" pos="0 0 0.07" material="dark_carbon"/>
        
        <!-- J2: Shoulder / Pitch (Realistic Mechanical Stop at -0.35 rad) -->
        <body name="left_link2" pos="0 0 0.15">
          <joint name="left_shoulder" type="hinge" axis="0 1 0" range="-0.35 1.8"/>
          <geom type="capsule" size="0.048 0.11" pos="0 0 0.11" material="matte_silver"/>
          <geom type="cylinder" size="0.05 0.015" pos="0 0 0.22" material="anodized_magenta" contype="0" conaffinity="0"/>

          <!-- J3: Shoulder Roll -->
          <body name="left_link3" pos="0 0 0.22">
            <joint name="left_shoulder_roll" type="hinge" axis="0 0 1" range="-2.2 2.2"/>
            <geom type="capsule" size="0.045 0.10" pos="0 0 0.10" material="dark_carbon"/>

            <!-- J4: Elbow / Pitch (High-speed Flick Joint) -->
            <body name="left_link4" pos="0 0 0.20">
              <joint name="left_elbow" type="hinge" axis="0 1 0" range="-2.2 2.4" damping="0.8"/>
              <geom type="capsule" size="0.04 0.11" pos="0 0 0.11" material="matte_silver"/>

              <!-- J5: Forearm Roll -->
              <body name="left_link5" pos="0 0 0.22">
                <joint name="left_forearm_roll" type="hinge" axis="0 0 1" range="-2.8 2.8"/>
                <geom type="cylinder" size="0.038 0.04" pos="0 0 0.04" material="dark_carbon"/>

                <!-- J6: Wrist Pitch -->
                <body name="left_link6" pos="0 0 0.08">
                  <joint name="left_wrist_pitch" type="hinge" axis="0 1 0" range="-1.6 1.6" damping="0.5"/>
                  <geom type="box" size="0.035 0.04 0.03" pos="0 0 0.03" material="anodized_magenta"/>

                  <!-- J7: Wrist Roll & Gripper Base -->
                  <body name="left_link7" pos="0 0 0.06">
                    <joint name="left_wrist_roll" type="hinge" axis="0 0 1" range="-3.14 3.14" damping="0.3"/>
                    <geom type="box" size="0.045 0.05 0.02" pos="0 0 0.02" material="dark_carbon"/>
                    <site name="left_hand_site" pos="0 0 0.10" size="0.015" material="neon_trajectory"/>
                    
                    <!-- Left Finger 1 -->
                    <body name="left_finger_left" pos="0 0.025 0.04">
                      <joint name="left_gripper_finger1" type="slide" axis="0 1 0" range="0.0 0.045" damping="2.0"/>
                      <geom type="box" size="0.01 0.008 0.04" pos="0 0 0.04" material="dark_carbon" friction="2.5 0.1 0.01"/>
                      <geom class="cushion_geom" type="box" size="0.008 0.004 0.035" pos="0 -0.006 0.04"/>
                    </body>
                    <!-- Left Finger 2 -->
                    <body name="left_finger_right" pos="0 -0.025 0.04">
                      <joint name="left_gripper_finger2" type="slide" axis="0 -1 0" range="0.0 0.045" damping="2.0"/>
                      <geom type="box" size="0.01 0.008 0.04" pos="0 0 0.04" material="dark_carbon" friction="2.5 0.1 0.01"/>
                      <geom class="cushion_geom" type="box" size="0.008 0.004 0.035" pos="0 0.006 0.04"/>
                    </body>
                    <!-- Left Camera (Wrist PiP) -->
                    <camera name="left_wrist_cam" pos="0.06 0 0.08" euler="0.3 0.8 1.57" fovy="65"/>
                  </body>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>

    <!-- ============================================================== -->
    <!-- RIGHT ARM: 7-DOF DYNAMIC CATCHER (Base at X = +0.50m)          -->
    <!-- ============================================================== -->
    <body name="right_base" pos="0.50 0 0.71">
      <geom name="right_pedestal" type="cylinder" size="0.07 0.05" pos="0 0 0.05" material="dark_carbon" solref="0.002 1" solimp="0.98 0.995 0.001"/>
      <geom name="right_accent" type="cylinder" size="0.072 0.005" pos="0 0 0.10" material="anodized_cyan" contype="0" conaffinity="0"/>
      
      <!-- J1: Waist / Yaw -->
      <body name="right_link1" pos="0 0 0.105">
        <joint name="right_waist" type="hinge" axis="0 0 1" range="-2.6 2.6"/>
        <geom type="cylinder" size="0.058 0.07" pos="0 0 0.07" material="dark_carbon"/>
        
        <!-- J2: Shoulder / Pitch (Realistic Mechanical Stop at -0.35 rad) -->
        <body name="right_link2" pos="0 0 0.15">
          <joint name="right_shoulder" type="hinge" axis="0 1 0" range="-0.35 1.8"/>
          <geom type="capsule" size="0.048 0.11" pos="0 0 0.11" material="matte_silver"/>
          <geom type="cylinder" size="0.05 0.015" pos="0 0 0.22" material="anodized_cyan" contype="0" conaffinity="0"/>

          <!-- J3: Shoulder Roll -->
          <body name="right_link3" pos="0 0 0.22">
            <joint name="right_shoulder_roll" type="hinge" axis="0 0 1" range="-2.2 2.2"/>
            <geom type="capsule" size="0.045 0.10" pos="0 0 0.10" material="dark_carbon"/>

            <!-- J4: Elbow / Pitch -->
            <body name="right_link4" pos="0 0 0.20">
              <joint name="right_elbow" type="hinge" axis="0 1 0" range="-2.2 2.4" damping="1.2"/>
              <geom type="capsule" size="0.04 0.11" pos="0 0 0.11" material="matte_silver"/>

              <!-- J5: Forearm Roll -->
              <body name="right_link5" pos="0 0 0.22">
                <joint name="right_forearm_roll" type="hinge" axis="0 0 1" range="-2.8 2.8"/>
                <geom type="cylinder" size="0.038 0.04" pos="0 0 0.04" material="dark_carbon"/>

                <!-- J6: Wrist Pitch -->
                <body name="right_link6" pos="0 0 0.08">
                  <joint name="right_wrist_pitch" type="hinge" axis="0 1 0" range="-1.6 1.6" damping="0.8"/>
                  <geom type="box" size="0.035 0.04 0.03" pos="0 0 0.03" material="anodized_cyan"/>

                  <!-- J7: Wrist Roll & Catcher End-Effector -->
                  <body name="right_link7" pos="0 0 0.06">
                    <joint name="right_wrist_roll" type="hinge" axis="0 0 1" range="-3.14 3.14" damping="0.5"/>
                    <geom type="box" size="0.045 0.06 0.02" pos="0 0 0.02" material="dark_carbon"/>
                    <site name="right_catch_center" pos="0 0 0.08" size="0.02" material="neon_target"/>
                    
                    <!-- High-compliance Soft Catching Paddle / Basket with Sensors -->
                    <geom class="cushion_geom" name="right_cushion_base" type="box" size="0.06 0.07 0.012" pos="0 0 0.04"/>
                    <geom class="cushion_geom" name="right_rim_fwd" type="box" size="0.06 0.008 0.025" pos="0 0.065 0.055"/>
                    <geom class="cushion_geom" name="right_rim_bwd" type="box" size="0.06 0.008 0.025" pos="0 -0.065 0.055"/>
                    <geom class="cushion_geom" name="right_rim_left" type="box" size="0.008 0.06 0.025" pos="-0.055 0 0.055"/>
                    <geom class="cushion_geom" name="right_rim_right" type="box" size="0.008 0.06 0.025" pos="0.055 0 0.055"/>

                    <!-- Right Camera (Wrist PiP) -->
                    <camera name="right_wrist_cam" pos="-0.06 0 0.08" euler="-0.3 -0.8 -1.57" fovy="65"/>
                  </body>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>

    <!-- ============================================================== -->
    <!-- BALLISTIC PROJECTILE (Target Object: Aerodynamic Capsule/Cube)  -->
    <!-- ============================================================== -->
    <body name="projectile" pos="-0.25 0 0.88">
      <freejoint name="projectile_joint"/>
      <geom name="projectile_geom" type="capsule" size="0.025 0.02" mass="0.08" material="projectile_mat" friction="1.8 0.05 0.005" solref="0.008 1" solimp="0.9 0.98 0.001"/>
      <site name="projectile_center" pos="0 0 0" size="0.01" material="neon_target"/>
    </body>

    <!-- Studio Overview Cameras (Widescreen Full Arena Coverage) -->
    <camera name="front_overview" pos="0 -1.8 1.4" euler="1.1 0 0" fovy="62"/>
    <camera name="side_cinematic" pos="-1.8 0 1.3" euler="0.8 -1.15 -1.57" fovy="58"/>
    <camera name="top_tactical" pos="0 0 2.5" euler="0 0 0" fovy="65"/>
  </worldbody>

  <!-- ================================================================ -->
  <!-- ACTUATORS: High-Speed Torque & Position Controlled Motors        -->
  <!-- ================================================================ -->
  <actuator>
    <!-- Left Arm Thrower Actuators -->
    <motor name="act_left_waist" joint="left_waist" gear="80" ctrlrange="-1 1"/>
    <motor name="act_left_shoulder" joint="left_shoulder" gear="100" ctrlrange="-1 1"/>
    <motor name="act_left_shoulder_roll" joint="left_shoulder_roll" gear="60" ctrlrange="-1 1"/>
    <motor name="act_left_elbow" joint="left_elbow" gear="70" ctrlrange="-1 1"/>
    <motor name="act_left_forearm_roll" joint="left_forearm_roll" gear="40" ctrlrange="-1 1"/>
    <motor name="act_left_wrist_pitch" joint="left_wrist_pitch" gear="30" ctrlrange="-1 1"/>
    <motor name="act_left_wrist_roll" joint="left_wrist_roll" gear="20" ctrlrange="-1 1"/>
    <position name="act_left_finger1" joint="left_gripper_finger1" kp="800" kv="40" ctrlrange="0 0.045"/>
    <position name="act_left_finger2" joint="left_gripper_finger2" kp="800" kv="40" ctrlrange="0 0.045"/>

    <!-- Right Arm Catcher Actuators -->
    <motor name="act_right_waist" joint="right_waist" gear="80" ctrlrange="-1 1"/>
    <motor name="act_right_shoulder" joint="right_shoulder" gear="100" ctrlrange="-1 1"/>
    <motor name="act_right_shoulder_roll" joint="right_shoulder_roll" gear="60" ctrlrange="-1 1"/>
    <motor name="act_right_elbow" joint="right_elbow" gear="70" ctrlrange="-1 1"/>
    <motor name="act_right_forearm_roll" joint="right_forearm_roll" gear="40" ctrlrange="-1 1"/>
    <motor name="act_right_wrist_pitch" joint="right_wrist_pitch" gear="30" ctrlrange="-1 1"/>
    <motor name="act_right_wrist_roll" joint="right_wrist_roll" gear="20" ctrlrange="-1 1"/>
  </actuator>

  <!-- ================================================================ -->
  <!-- SENSORS: Telemetry and Dynamic Contact Interception              -->
  <!-- ================================================================ -->
  <sensor>
    <touch name="sensor_catcher_touch" site="right_catch_center"/>
    <framepos name="sensor_projectile_pos" objtype="site" objname="projectile_center"/>
    <framelinvel name="sensor_projectile_vel" objtype="site" objname="projectile_center"/>
    <framepos name="sensor_left_hand_pos" objtype="site" objname="left_hand_site"/>
    <framepos name="sensor_right_catch_pos" objtype="site" objname="right_catch_center"/>
  </sensor>

  <!-- ================================================================ -->
  <!-- CONTACT: Collision Filtering & Smooth Bearing Motion             -->
  <!-- ================================================================ -->
  <contact>
    <exclude body1="left_base" body2="left_link1"/>
    <exclude body1="right_base" body2="right_link1"/>
    <exclude body1="left_link1" body2="left_link2"/>
    <exclude body1="right_link1" body2="right_link2"/>
  </contact>
</mujoco>
"""

with open("assets/aloha_bimanual_toss.xml", "w", encoding="utf-8") as f:
    f.write(xml_content)

model = mujoco.MjModel.from_xml_path("assets/aloha_bimanual_toss.xml")
data = mujoco.MjData(model)
mujoco.mj_step(model, data)
print(f"물리 모델 탑재 완료! (관절 {model.njnt}개, 모터/서보 {model.nu}개, 상태 차원 {model.nq})")


---  
## 🧠 Step 3. 4단계 위상 기반 물리 보상 엔진 (Phase-Aware Reward Engine)

### 📖 왜 이 단계를 수행하나요?
공중에 던져서 받는 고난도 작업은 "마지막에 잡았을 때만 보상"을 주면 성공 확률이 0%에 수렴하는 **희소 보상(Sparse Reward)** 문제가 발생합니다.
따라서 연속적인 물리 과정을 **4개의 명확한 위상(Phase)**으로 나누어 즉각적인 가이드 피드백을 제공합니다:

1. **Phase 0 (Grasp):** 캡슐에 손을 뻗어 단단히 쥐는 행동 유도
2. **Phase 1 (Toss):** 최적의 포물선 정점($Z=1.25\text{m}$)을 향해 가속 투척 및 릴리즈 유도
3. **Phase 2 (Flight):** 날아오는 탄도 궤적을 예측하여 캐칭 패들을 사전 착지점에 신속 배치
4. **Phase 3 (Cushion):** 접촉 순간 모터의 역토크로 충격을 완충 흡수하여 튕겨나가지 않도록 안착 (+50 대형 보너스)

👉 **[실행 방법]** 아래 셀을 실행하여 보상 엔진 함수를 정의합니다.

In [ ]:
class PhaseRewardEngine:
    def __init__(self):
        self.toss_launch_pos = np.array([-0.25, 0.0, 0.88])
        self.apex_target_pos = np.array([0.0, 0.0, 1.25])
        self.catch_target_pos = np.array([0.25, 0.0, 0.88])
        self.current_phase = 0

    def reset(self):
        self.current_phase = 0

    def compute(self, env_data):
        proj_pos = env_data["projectile_pos"]
        proj_vel = env_data["projectile_vel"]
        left_pos = env_data["left_hand_pos"]
        right_pos = env_data["right_catch_pos"]
        gripper_w = env_data["left_gripper_width"]
        has_contact = env_data["catcher_contact"]
        torques = env_data["torques"]

        reward = 0.0
        info = {"phase": self.current_phase, "is_success": False}
        dist_left = np.linalg.norm(left_pos - proj_pos)
        dist_catch = np.linalg.norm(proj_pos - right_pos)

        # Phase 0: Grasp
        if self.current_phase == 0:
            reward += 2.0 * np.exp(-15.0 * dist_left)
            if dist_left < 0.04 and gripper_w < 0.025:
                reward += 5.0
                self.current_phase = 1

        # Phase 1: Toss
        elif self.current_phase == 1:
            dist_to_apex = np.linalg.norm(proj_pos - self.apex_target_pos)
            vel_x_reward = np.clip(proj_vel[0], 0.0, 2.5) * 1.5
            vel_z_reward = np.clip(proj_vel[2], 0.0, 2.0) * 1.2
            reward += vel_x_reward + vel_z_reward + 2.0 * np.exp(-5.0 * dist_to_apex)
            if dist_left > 0.08 and proj_vel[0] > 0.5:
                reward += 10.0
                self.current_phase = 2

        # Phase 2: Flight & Analytical Intercept Guidance
        elif self.current_phase == 2:
            g = 9.81
            z_catch = 0.88
            vz = proj_vel[2]
            disc = vz**2 + 2.0 * g * max(0.0, proj_pos[2] - z_catch)
            t_impact = (vz + np.sqrt(disc)) / g if disc >= 0 else 0.1
            t_impact = np.clip(t_impact, 0.0, 1.2)
            predicted_landing = np.array([
                proj_pos[0] + proj_vel[0] * t_impact,
                proj_pos[1] + proj_vel[1] * t_impact,
                z_catch
            ])
            dist_to_pred = np.linalg.norm(right_pos - predicted_landing)
            reward += 4.0 * np.exp(-10.0 * dist_to_pred) + 2.5 * np.exp(-6.0 * dist_catch)
            if proj_pos[0] > -0.1 and proj_pos[0] < 0.2 and proj_pos[2] > 1.05:
                reward += 2.0
            if dist_catch < 0.12 or has_contact:
                reward += 10.0
                self.current_phase = 3

        # Phase 3: Cushion Catch
        elif self.current_phase == 3:
            reward += 5.0 * np.exp(-15.0 * dist_catch)
            if has_contact:
                proj_speed = np.linalg.norm(proj_vel)
                reward += 15.0 + 5.0 * np.exp(-3.0 * proj_speed)
                if dist_catch < 0.05 and proj_speed < 0.25:
                    reward += 50.0
                    info["is_success"] = True

        reward -= 0.0005 * np.sum(np.square(torques))
        info["phase"] = self.current_phase
        return reward, info

print("고도화된 탄도 예측 보상 엔진 (PhaseRewardEngine) 탑재 완료!")


---  
## 🎮 Step 4. 순수 물리 연산 전용 헤드리스 환경 (`AlohaTossCatchEnv`) 조립

### 📖 왜 이 단계를 수행하나요?
화면 렌더링 부하를 일체 없앤 **순수 수학/동역학 연산 전용 Gymnasium 환경**입니다.

* **관측 공간 (55차원):** 관절 각도/속도(32개) + 비행체 6D 상태(13개) + 상대 거리 벡터(6개) + 위상 원-핫(4개)
* **행동 공간 (16차원):** 16개 관절 모터의 토크 제어
* **역운동학(IK) 초기화:** 시작하자마자 공을 쥔 최적의 자세에서 출발하여 불필요한 공 탐색 시간을 단축
* **도메인 랜덤화:** 물체 초기 위치에 미세 노이즈($\pm 1\text{cm}$)를 주어 실제 물리 세계에서도 강건하게 작동하도록 유도

👉 **[실행 방법]** 아래 셀을 실행하여 환경 클래스를 메모리에 로드합니다.

In [ ]:
from gymnasium import spaces

class AlohaTossCatchEnv(gym.Env):
    def __init__(self, xml_path="assets/aloha_bimanual_toss.xml", frame_skip=10, max_episode_steps=250):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)
        self.frame_skip = frame_skip
        self.max_episode_steps = max_episode_steps
        self.current_step = 0

        self.site_left = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, "left_hand_site")
        self.site_right = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, "right_catch_center")
        self.site_proj = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, "projectile_center")

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.model.nu,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(55,), dtype=np.float32)
        self.reward_engine = PhaseRewardEngine()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.reward_engine.reset()
        mujoco.mj_resetData(self.model, self.data)

        # 정밀 역운동학(IK) 시작 자세
        self.data.qpos[0] = 0.0; self.data.qpos[1] = 0.866; self.data.qpos[3] = 2.138; self.data.qpos[5] = 1.050
        self.data.qpos[7] = 0.012; self.data.qpos[8] = 0.012
        self.data.qpos[9] = 0.0; self.data.qpos[10] = -0.908; self.data.qpos[12] = -2.176; self.data.qpos[14] = -1.009
        
        # 물체 위치 도메인 랜덤화
        self.data.qpos[16] = -0.25 + self.np_random.uniform(-0.01, 0.01)
        self.data.qpos[17] = 0.0 + self.np_random.uniform(-0.01, 0.01)
        self.data.qpos[18] = 0.88
        self.data.qpos[19:23] = [1.0, 0.0, 0.0, 0.0]
        self.data.qvel[:] = 0.0
        mujoco.mj_forward(self.model, self.data)
        return self._get_obs(), {"phase": 0}

    def step(self, action):
        self.current_step += 1
        action = np.clip(action, -1.0, 1.0)
        ctrl = np.zeros(self.model.nu, dtype=np.float64)
        ctrl[:7] = action[:7]
        ctrl[7] = (action[7] + 1.0) * 0.5 * 0.045
        ctrl[8] = (action[8] + 1.0) * 0.5 * 0.045
        ctrl[9:16] = action[9:16]
        self.data.ctrl[:] = ctrl

        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)

        left_pos = self.data.site_xpos[self.site_left].copy()
        right_pos = self.data.site_xpos[self.site_right].copy()
        proj_pos = self.data.site_xpos[self.site_proj].copy()
        proj_vel = self.data.qvel[16:19].copy()
        has_contact = bool(self.data.sensordata[0] > 0.05)
        gripper_w = float(self.data.qpos[7] + self.data.qpos[8])

        env_data = {
            "projectile_pos": proj_pos, "projectile_vel": proj_vel,
            "left_hand_pos": left_pos, "right_catch_pos": right_pos,
            "left_gripper_width": gripper_w, "catcher_contact": has_contact,
            "torques": self.data.actuator_force.copy()
        }
        reward, info = self.reward_engine.compute(env_data)
        terminated = False; truncated = False
        if proj_pos[2] < 0.68:
            reward -= 20.0; terminated = True; info["reason"] = "dropped"
        if info.get("is_success", False):
            terminated = True; info["reason"] = "success"
        if self.current_step >= self.max_episode_steps:
            truncated = True; info["reason"] = "timeout"
        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self):
        l_qp, l_qv = self.data.qpos[:9], self.data.qvel[:9]
        r_qp, r_qv = self.data.qpos[9:16], self.data.qvel[9:16]
        p_pos, p_vel = self.data.site_xpos[self.site_proj], self.data.qvel[16:19]
        p_quat, p_ang = self.data.qpos[19:23], self.data.qvel[19:22]
        l_to_p = p_pos - self.data.site_xpos[self.site_left]
        p_to_r = self.data.site_xpos[self.site_right] - p_pos
        phase_oh = np.zeros(4, dtype=np.float32)
        phase_oh[self.reward_engine.current_phase] = 1.0
        return np.concatenate([l_qp, l_qv, r_qp, r_qv, p_pos, p_vel, p_quat, p_ang, l_to_p, p_to_r, phase_oh]).astype(np.float32)

test_env = AlohaTossCatchEnv()
obs, info = test_env.reset()
print(f"✅ 헤드리스 환경 초기화 완료! 관측 크기: {obs.shape}, 행동 공간: {test_env.action_space.shape}")

---  
## 🚀 Step 5. 멀티 프로세스 병렬 PPO 대규모 강화학습 (High-Speed Training)

### 📖 왜 이 단계를 수행하나요?
코랩 클라우드의 진정한 강점은 **CPU 멀티코어 병렬화**입니다.

* `DummyVecEnv` 또는 다중 환경을 묶어 시뮬레이터를 병렬로 실행합니다.
* 화면 렌더링이 전혀 없으므로 CPU/GPU가 100% 동역학 계산과 신경망 역전파에만 집중되어 **초당 수천~수만 FPS**의 경이로운 속도로 훈련됩니다.
* 100,000 ~ 200,000 스텝의 대규모 학습이 단 1~3분 만에 완료됩니다.
* 터미널 로그의 `ep_rew_mean`(평균 누적 보상)이 음수에서 양수로 급상승하는 수렴 과정을 지켜보세요.

👉 **[실행 방법]** 아래 셀을 실행하여 고속 PPO 학습을 시작합니다.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

def make_env(rank):
    def _init():
        return Monitor(AlohaTossCatchEnv())
    return _init

# 8개 병렬 환경 벡터화 (초당 15,000+ FPS 고속 연산)
train_env = DummyVecEnv([make_env(i) for i in range(8)])

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.002,
    policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
    verbose=1
)

# 400,000 스텝 대규모 PPO 학습 (8병렬 기준 약 2~3분 소요)
print("🏋️ [코랩 고속 모드] 8개 병렬 환경에서 대규모 PPO 정책 학습을 시작합니다...")
model.learn(total_timesteps=400000)

os.makedirs("checkpoints", exist_ok=True)
model.save("checkpoints/aloha_toss_catch_final.zip")
model.save("checkpoints/aloha_toss_catch_ppo.zip")
print("\n🎉 학습 완벽 종료! 똑똑해진 두뇌 파일 저장 완료: checkpoints/aloha_toss_catch_final.zip")


---  
## 💾 Step 6. 훈련된 최적 정책 가중치(`.zip`) 내 컴퓨터로 다운로드

### 📖 왜 이 단계를 수행하나요?
클라우드 훈련의 목표는 **완성된 인공지능 두뇌 파일(`aloha_toss_catch_ppo.zip`, 약 3MB)**을 수확하는 것입니다.
이 셀을 실행하면 학습된 가중치 파일이 브라우저를 통해 내 로컬 PC의 `다운로드` 폴더로 즉시 전송됩니다.

### 🎬 로컬 PC에서 60fps 콕핏 HUD 영상 뽑는 법:
1. 다운로드받은 `aloha_toss_catch_ppo.zip` 파일을 로컬 프로젝트의 `checkpoints/` 폴더에 넣습니다.
2. 로컬 터미널(PowerShell)에서 아래 명령어를 실행하면, 내 컴퓨터가 60fps 콕핏 HUD를 합성하여 최고 화질의 GIF/영상을 추출합니다:
   ```powershell
   python evaluate.py --model checkpoints/aloha_toss_catch_ppo.zip --output assets/colab_trained_demo.gif
   ```

👉 **[실행 방법]** 아래 셀을 실행하세요. 브라우저 하단에 다운로드 창이 열립니다.

In [ ]:
try:
    from google.colab import files
    print("📥 학습된 최적 두뇌 파일 (aloha_toss_catch_final.zip) 다운로드를 시작합니다...")
    files.download("checkpoints/aloha_toss_catch_final.zip")
    print("✅ 브라우저 다운로드 창이 열렸습니다! 다운로드된 파일을 로컬 PC의 checkpoints/ 폴더에 넣어주세요.")
except ImportError:
    print("💡 로컬 환경에서 실행 중이므로 다운로드 명령을 건너뜁니다.")
